# Lab 4 — Intervals and Tests on the Frontier

**MA553 · Session 5**

Name:

---

In Lab 3 you compared five model families on one test set of 100,000 trips.
Each comparison gave one number per model. In this lab you measure how far
those numbers could move, and you test whether the differences between models
are more than chance.

**Part A** practises the methods of the session: the sampling distribution of
a mean, a 95% confidence interval, a bootstrap interval for a test MAE, and
two tests that compare two models on the same trips.

**Part B** applies them to results from Lab 3 and asks what they show.

Work through the cells in order. Six cells are marked **Write this yourself**:
A2, A4, A5, B1, B2 and B3. Each says which variables to create; later cells use
those names. The **Answer here** cells are short written answers.

# Part A — The methods

## The data and the models

This lab uses the January 2025 file you downloaded for Lab 2. The cell below
looks for it in `../lab2/data/`. If it is missing, run `python get_data.py` in
the `lab2` folder first.

The filter, the test set and the training sample are the same as in Lab 3.
Therefore the test MAEs are the same as in your Lab 3 frontier table.

Run this.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

PATH = Path("../lab2/data/yellow_tripdata_2025-01.parquet")
COLUMNS = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance",
           "PULocationID", "DOLocationID"]
trips = pd.read_parquet(PATH, columns=COLUMNS)
print(f"Trips in the file: {len(trips):,}")

## A1 — The population, the test set and four models

This cell repeats the Lab 3 preparation. It then fits four models on the same
1,000,000 training trips: the decision tree, the random forest and gradient
boosting from Lab 3 A3, and the linear model with hour one-hot and zone pairs
from Lab 3 B2.

For every test trip and every model it stores the **absolute error**, the size
of the mistake in minutes. The table `errors` has one row per test trip and
one column per model. The mean of a column is that model's test MAE. The
columns are named `tree` (decision tree), `forest` (random forest), `boosting`
(gradient boosting) and `linear_pairs` (linear model with zone pairs).

The variable `population` holds the duration of every kept trip. Part A uses
it as the population from which samples are drawn.

The random forest takes one to two minutes. Run this once.

In [ ]:
trips["minutes"] = (trips["tpep_dropoff_datetime"] - trips["tpep_pickup_datetime"]).dt.total_seconds() / 60
keep = trips["minutes"].between(1, 120) & (trips["trip_distance"] > 0) & (trips["trip_distance"] < 60)
trips = trips[keep].copy()
trips["hour"] = trips["tpep_pickup_datetime"].dt.hour
trips["weekday"] = trips["tpep_pickup_datetime"].dt.weekday
FEATURES = ["trip_distance", "hour", "weekday", "PULocationID", "DOLocationID"]

test = trips.sample(100_000, random_state=1)
pool = trips.drop(test.index)
for table in (pool, test):
    table["pair"] = table["PULocationID"].astype(str) + "-" + table["DOLocationID"].astype(str)
train = pool.sample(1_000_000, random_state=2)

population = trips["minutes"].to_numpy()
print(f"Population: {len(population):,} trips, mean duration {population.mean():.2f} minutes")


def linear_pairs():
    prepare = ColumnTransformer([
        ("numbers", StandardScaler(), ["trip_distance"]),
        ("categories", OneHotEncoder(handle_unknown="ignore"), ["hour", "weekday", "PULocationID", "DOLocationID"]),
        ("pairs", OneHotEncoder(handle_unknown="ignore", min_frequency=5), ["pair"]),
    ])
    return make_pipeline(prepare, Ridge(alpha=1.0))


models = {
    "tree": (DecisionTreeRegressor(max_depth=12, random_state=0), FEATURES),
    "forest": (RandomForestRegressor(n_estimators=100, max_depth=12, n_jobs=-1, random_state=0), FEATURES),
    "boosting": (HistGradientBoostingRegressor(max_iter=300, random_state=0), FEATURES),
    "linear_pairs": (linear_pairs(), FEATURES + ["pair"]),
}
errors = pd.DataFrame(index=test.index)
for name, (model, columns) in models.items():
    model.fit(train[columns], train["minutes"])
    errors[name] = np.abs(test["minutes"].to_numpy() - model.predict(test[columns]))

print(errors.mean().round(2).rename("test MAE (minutes)"))

Expected: population mean 14.71 minutes; test MAE decision tree
3.22, random forest 3.14, gradient boosting 2.89,
linear with zone pairs 3.36.

## A2 — The sampling distribution of a mean

**Write this yourself.**

The **sampling distribution** of a statistic is its distribution across many
samples. Its standard deviation is the **standard error**.

What to do:

1. Create a random generator: `rng = np.random.default_rng(553)`. Use `rng`
   for every random draw in this lab.
2. Draw 1,000 samples of 200 trips from `population`, without replacement
   (`rng.choice(population, 200, replace=False)`). Store the 1,000 sample means
   in an array named `means`.
3. Plot a histogram of `means`, with a vertical line at the population mean.
   Label both axes.
4. Compute `se_sampling`, the standard deviation of `means`.
5. Draw one more sample of 200 trips, named `one`. Compute `se_formula`, the
   standard error estimated from that sample alone: `one.std(ddof=1) / np.sqrt(200)`.
6. Print `se_sampling` and `se_formula`.

In [ ]:
# Your code here.


Expected: a standard deviation of about 0.75 minutes
for the 1,000 means. The value from one sample depends on which 200 trips were
drawn, and it can differ from this by 0.1 minutes or more.

## A3 — A 95% confidence interval and its coverage

The 95% confidence interval for a mean is the sample mean ± 1.96 standard
errors. This cell computes the interval from `one`, then repeats the procedure
on 100 new samples and counts how many intervals contain the population mean.
Run this.

In [ ]:
low = one.mean() - 1.96 * se_formula
high = one.mean() + 1.96 * se_formula
print(f"95% interval from one sample: {low:.2f} to {high:.2f} minutes")

contains = 0
for _ in range(100):
    sample = rng.choice(population, 200, replace=False)
    half = 1.96 * sample.std(ddof=1) / np.sqrt(200)
    contains += (sample.mean() - half) <= population.mean() <= (sample.mean() + half)
print(f"Intervals that contain the population mean: {contains} of 100")

Expected: close to 95 of 100. The count depends on the 100 samples drawn.
Values from about 88 to 99 are normal (the solution run gives 89).

## A4 — A bootstrap interval for a test MAE

**Write this yourself.**

The test MAE is a mean of absolute errors, so it also has an interval. The
**bootstrap** gives one without a formula: resample the values with
replacement, recompute the mean, repeat, and keep the middle 95 of every 100
results. This is the **percentile bootstrap interval**.

What to do:

1. Write a function `bootstrap_ci(values, reps=1000)` that returns the 2.5th
   and 97.5th percentiles of `reps` bootstrap means of `values` (a NumPy
   array). Draw each bootstrap sample with
   `values[rng.integers(0, len(values), len(values))]`.
2. Apply it to the gradient boosting errors (column `boosting`) on all 100,000
   test trips. Store the result as `ci_100k`.
3. Create `small = errors.sample(1000, random_state=5)`, a table of 1,000 test
   trips. Apply `bootstrap_ci` to its gradient boosting errors. Store the result as `ci_1k`.
4. Print both intervals with the test MAE they belong to.

In [ ]:
# Your code here.


Expected: on 100,000 trips, an interval about
0.05 minutes wide; on
1,000 trips, an interval about ten times wider.

## A5 — Comparing two models: the paired t-test and the swap test

**Write this yourself.**

The random forest and gradient boosting are scored on the same test trips. The
**paired difference** of a trip is its random forest error minus its gradient
boosting error. The null hypothesis is that, on all trips, the two models have
the same MAE.

The swap test (also called a permutation test) follows the session slides. If
the null hypothesis were true, it would not matter which model made which of a
trip's two errors. One **round** of swaps is:

1. For each trip, flip a coin. On heads, swap the trip's two errors. This flips
   the sign of its paired difference. On tails, keep them.
2. Compute the mean of the paired differences after the flips. This is one
   **swapped mean**.

The p-value is the number of swapped means at least as far from 0 as the
observed mean difference, on either side, divided by the number of rounds.

What to do:

1. Compute `d`, the array of 100,000 paired differences, and print its mean.
2. Run the paired t-test with `stats.ttest_rel(errors["forest"], errors["boosting"])`.
   Print the t statistic and the p-value.
3. Run 10,000 rounds of swaps. In code, one round multiplies `d` by random signs
   (`rng.choice([-1, 1], size=len(d))`) and takes the mean. Store the 10,000
   swapped means in `swapped`. To save memory, draw the signs for 250 rounds at
   a time.
4. Count the swapped means at least as far from 0 as the observed mean
   (`np.abs(swapped) >= abs(d.mean())`). Store the count as `n_extreme` and
   `p_perm = n_extreme / 10_000`. Print both. If the count is 0, the p-value is
   below 1 in 10,000.

In [ ]:
# Your code here.


Expected: a mean paired difference of 0.25 minutes
and t close to 51. The t-test may print `p = 0`. The
p-value is then smaller than the computer can represent, not exactly 0. The swap
test finds 0 of 10,000 swapped means as far from 0 as the observed one.

**Answer here.** Do the two tests agree? Is the difference statistically
significant at the 0.05 level? State the mean paired difference and its 95%
interval (mean ± 1.96 × standard error), and say what the interval adds to the
p-value.

*Your answer:*

# Part B — What the intervals and tests show

## B1 — The cost of Budget 2

**Write this yourself.**

In Lab 3, Budget 2 required a model whose predictions are a sum of per-feature
contributions. The best such model was the linear model with hour one-hot and
zone pairs. Its test MAE was higher than gradient boosting's. The difference
was the cost of the requirement.

What to do:

1. Compute the paired differences `d_budget`: the `linear_pairs` error minus
   the gradient boosting (`boosting`) error, per test trip.
2. Compute the 95% interval for their mean (mean ± 1.96 × standard error) and
   store it as `ci_budget`.
3. Run the paired t-test and print the t statistic and the p-value.

In [ ]:
# Your code here.


**Answer here.** Is the cost of Budget 2 statistically significant at the 0.05
level? Using the interval, state how large the cost could plausibly be.

*Your answer:*

## B2 — The same comparison on fewer trips

**Write this yourself.**

The decision tree and the random forest differ by a small amount. Compare them
on all 100,000 test trips and on the 1,000 trips in `small` (from A4).

What to do:

1. For each of the two tables, compute the mean of the paired differences
   (decision tree error minus random forest error), its 95% interval, and the
   p-value of the paired t-test.
2. Print one line per table.
3. Repeat the comparison on 200 random subsets of 1,000 test trips (draw each
   with `rng.choice(len(errors), 1000, replace=False)`). Count how many of the
   200 intervals exclude 0, and store the share as `share_excluded`.

In [ ]:
# Your code here.


**Answer here.**

1. On 1,000 trips, how often does the test detect the difference? Explain the
   difference from 100,000 trips, using the standard error. When the test does
   not detect it, which kind of error is that?
2. The difference on 100,000 trips is significant. Is it large enough to matter
   to a passenger? Give a reference point.

*Your answer:*

## B3 — A t-test on cross-validation folds

**Write this yourself.**

A common shortcut compares two models by their fold scores from
cross-validation. This cell checks what that shortcut does.

What to do:

1. Take 100,000 training trips: `cv_train = pool.sample(100_000, random_state=3)`.
2. Compute the five fold scores of the random forest and of gradient boosting with
   `cross_val_score`, using `KFold(5, shuffle=True, random_state=0)` and
   `scoring="neg_mean_absolute_error"`. Use the same settings as in A1. Multiply
   the scores by −1 so that they are MAEs.
3. Run `stats.ttest_rel` on the two arrays of five fold scores. Print the
   fold scores and the p-value.

In [ ]:
# Your code here.


**Answer here.** Explain why this p-value is not valid, even though the code
runs. Which assumption of the t-test do the five fold scores break?

*Your answer:*

## B4 — One limitation

**Answer here.** Name one thing the intervals and tests in this lab do not
establish, and what would be needed to establish it.

*Your answer:*

## Before you submit

- Restart the kernel and run every cell from top to bottom. A1 and B3 take a
  few minutes; that is expected.
- Check that every **Answer here** cell is filled in.
- Do not submit the data file.
- If you used an AI assistant, add a short note at the end saying what it
  produced and what you changed. You must be able to explain every line.